# 📗 LoRA: 어댑터만 학습해 노무 상담 모델 만들기

**LoRA**(Low-Rank Adaptation, 저랭크 적응)는 모델 가중치는 그대로 두고 **선형 층마다 붙인 작은 행렬(어댑터)만 학습**하는 파인튜닝 방법입니다. 지난 시간 풀 파인튜닝과 같은 모델(`Qwen/Qwen3.5-0.8B`)과 노무 상담 데이터로 어댑터를 학습해 저장합니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="교안 01에서 배울 네 가지: 1 LoRA 원리(큰 가중치는 그대로, 작은 어댑터만 학습), 2 어댑터 붙이기, 3 에폭마다 저장하며 학습, 4 결과 확인과 다시 쓰기(답 비교, 끄고 켜기, 다시 얹기).">

**학습 목표** (그림의 번호와 같습니다)

1. **LoRA가 $W$는 고정하고 작은 두 행렬 $B$, $A$만 학습하는 원리를 설명하고, 선형 층 하나의 학습 파라미터 수를 계산합니다.**
2. **`LoraConfig`로 `r`, `lora_alpha`, `target_modules`를 정하고, `get_peft_model`로 어댑터를 붙여 학습 파라미터 수를 확인합니다.**
3. **지난 시간과 같은 `SFTTrainer`로 어댑터를 lr 2e-4로 4에폭 학습하며 에폭마다 저장합니다.**
4. **같은 질문으로 학습 전·후 답을 비교하고, 저장한 어댑터를 새 베이스 모델에 다시 불러와 같은 답이 나오는지 확인합니다.**

## 1. PEFT와 LoRA

**PEFT(Parameter-Efficient Fine-Tuning, 파라미터 효율 파인튜닝)**
- 베이스 모델의 가중치는 고정하고, 따로 둔 적은 수의 파라미터만 학습하는 파인튜닝 방법을 통틀어 부르는 이름입니다. Hugging Face `peft` 라이브러리가 제공합니다.
- **베이스 모델:** 어댑터를 얹는 원래 모델입니다(여기서는 `Qwen/Qwen3.5-0.8B`).
- **목적:** 학습하는 파라미터가 적어 GPU 메모리가 적게 들고, 저장 파일도 작습니다. 상담 분야가 노무, 전월세 둘이면 모델 전체를 두 벌 저장하지 않고 베이스 모델 하나와 분야별 어댑터 파일 두 개를 둡니다(이 교안의 노무 어댑터 파일은 20.7MB, 지난 시간 풀 파인튜닝 체크포인트는 약 3GB).

**LoRA(Low-Rank Adaptation, 저랭크 적응)**
- 선형 층의 가중치 $W$는 고정하고, $W$가 바뀔 양을 작은 두 행렬의 곱 $BA$로 학습하는 PEFT 방법입니다([Hu et al., 2021](https://arxiv.org/abs/2106.09685)). 학습한 $B$, $A$를 어댑터(adapter)라고 부릅니다.
- **rank(랭크):** 행렬에서 서로 독립인 행의 최대 개수입니다. 다른 행을 몇 배 하거나 더해서 만들 수 있는 행은 세지 않습니다.
- **`r`:** $B$와 $A$를 곱할 때 맞붙는 **가운데 차원**입니다($B$는 $d_{out} \times$ **`r`**, $A$는 **`r`** $\times d_{in}$). `LoraConfig`에서 사람이 정하는 값입니다(이 교안 `r=8`). 곱 $BA$의 모양은 `r`과 상관없이 $W$와 같지만, $BA$의 행은 모두 $A$의 `r`개 행을 섞어 만들어지므로 rank가 `r` 이하입니다.
- **저랭크(low-rank):** rank가 낮다는 뜻입니다. 파인튜닝으로 $W$가 바뀌는 양은 rank가 낮은 행렬로도 충분히 나타낼 수 있다는 가설에서 LoRA가 출발했습니다.

<img src="images/06_rank_example.png" width="1050" alt="가운데 차원 r = 2인 B(4 × r)와 A(r × 4)를 곱한 BA(4 × 4)에서 1행과 2행은 독립인 행이고, 3행은 1행 + 2행, 4행은 1행 × 2로 만들 수 있어 rank는 2 = r입니다. BA의 모양은 r과 상관없이 4 × 4이지만 행은 모두 A의 r개 행을 섞어 만들므로 rank는 r 이하입니다(r은 우리가 정함, 이 교안 r = 8).">

<img src="images/01_full_vs_lora.png" width="1050" alt="풀 파인튜닝은 W 전체를 학습하고, LoRA는 W를 고정한 채 B와 A로 만든 '바뀔 양'만 학습합니다. 층 하나(3584 × 1024)에서 학습하는 수는 3,670,016개 대 36,864개(약 1%)입니다.">

LoRA 층(어댑터가 붙은 선형 층)은 고정한 $W$의 출력에 어댑터 출력을 더해 내보냅니다.

$\displaystyle h = W x + \frac{\alpha}{r}\, B A x$

- **$W$:** 베이스 모델의 가중치 행렬입니다. 모양은 $d_{out} \times d_{in}$(출력 크기 × 입력 크기)이고 학습하지 않습니다.
- **$A$, $B$:** 어댑터의 두 행렬로, 둘 다 학습합니다. $A$는 $r \times d_{in}$, $B$는 $d_{out} \times r$입니다.
- **$A$는 좁힘, $B$는 넓힘:** $A$는 길이 $d_{in}$인 입력 $x$를 $r$개 값으로 줄이고, $B$는 그 $r$개 값을 다시 $d_{out}$개로 늘립니다. 아래 `gate_proj` 예(r = 8)로는 1024개 → 8개 → 3584개입니다. 그래서 어댑터 출력 $BAx$는 $Wx$와 길이가 같아 둘을 더할 수 있습니다.
- **$r$과 $\alpha$:** 둘 다 **하이퍼파라미터**(학습 전에 사람이 정하고, 학습 중에는 바뀌지 않는 값)입니다. `LoraConfig`의 `r`과 `lora_alpha`로 정합니다.
- **배율 $\alpha / r$:** 어댑터 출력에 곱하는 고정된 수입니다(peft 안의 이름은 `scaling`).
- **파라미터:** 학습하면서 값이 바뀌는 수입니다. 여기서는 $A$, $B$의 원소뿐입니다.
- **처음 값:** $A$는 작은 무작위 값, $B$는 0에서 시작합니다. $BA = 0$이라 어댑터를 붙인 직후 모델 출력은 베이스 모델과 같습니다.

<img src="images/02_lora_decomposition.png" width="1050" alt="LoRA 층의 계산: 입력 x가 고정한 W 길과 A(좁힘), B(넓힘), 배율 α / r 길로 나뉘었다가 더해져 h = W x + (α / r) B A x가 됩니다. B가 0에서 시작해 붙인 직후에는 베이스 모델과 같습니다.">

MLP(트랜스포머 단원의 FFN, 모듈 이름 `mlp`) 안의 선형 층 `gate_proj` 하나로 계산해 봅니다. $W$는 3584 × 1024($d_{out}$ 3584, $d_{in}$ 1024)입니다.

1. **풀 파인튜닝:** $W$의 3584 × 1024 = 3,670,016개를 모두 학습합니다.
2. **LoRA(r = 8):** $B$의 3584 × 8 = 28,672개와 $A$의 8 × 1024 = 8,192개, 합 36,864개만 학습합니다. 식으로는 $r \times (d_{out} + d_{in})$입니다.
3. **비율:** 36,864 / 3,670,016 = 약 1.0%입니다. `r`을 16으로 두 배 하면 학습 파라미터 수도 두 배(73,728개)입니다.
4. **기울기와 AdamW 상태:** 고정한 $W$에는 생기지 않고 어댑터 몫만 생깁니다(메모리 계산은 다음 시간).

**풀 파인튜닝과 LoRA, 언제 무엇을 쓸까**

| 항목 | 풀 파인튜닝 | LoRA |
|---|---|---|
| 학습하는 것 | 가중치 전체 | 어댑터(베이스 가중치 수의 1% 미만) |
| 저장 | 모델 전체 | 어댑터 파일만 |
| 상담 분야가 여럿일 때(노무, 전월세) | 분야마다 모델 전체 | 베이스 모델 하나 + 분야별 어댑터 |

- **LoRA:** GPU 메모리가 적을 때, 분야가 여럿일 때, 이 단원처럼 질문-답변 SFT일 때 씁니다. [Biderman et al., 2024](https://arxiv.org/abs/2405.09673)는 코드, 수학 학습 실험으로 LoRA가 학습 분야 밖의 원래 능력을 풀 파인튜닝보다 덜 잃는다는 것을 보였습니다.
- **풀 파인튜닝:** 새 지식을 대량으로 넣고 GPU가 충분할 때 씁니다(예: 한 분야의 문서를 통째로 학습시키는 경우). 어댑터는 파라미터가 적어(`r`이 작을수록 더 적음) 담을 수 있는 정보에 한계가 있습니다. [LoRA Without Regret](https://thinkingmachines.ai/blog/lora/) 실험에서는 학습 데이터가 그 한계를 넘을 만큼 크면 LoRA가 풀 파인튜닝보다 학습 효율이 떨어졌습니다.

## 2. 실습 준비

### 2-1. 라이브러리와 경로

지난 시간과 같은 준비 셀입니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 2-2. 학습 전 모델

풀 파인튜닝과 같은 조건으로 비교하려고, 지난 시간과 같은 모델을 같은 방법(float32)으로 불러옵니다.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저(문장 <-> 토큰 번호 변환)와 학습 전 모델(Base)을 지난 시간과 같은 방법(float32)으로 불러옵니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# 전체 파라미터 수: 풀 파인튜닝은 이 전부를 학습했고, LoRA는 이 전부를 고정한 채 어댑터만 더해 학습합니다
print(f"베이스 모델 파라미터 수: {base_model.num_parameters():,}")

### 2-3. 지난 시간과 같은 함수와 데이터

`generate_reply`, `load_jsonl`, `to_prompt_completion`, 시스템 프롬프트, 학습·검증 파일이 지난 시간 교안 01과 같습니다. 그래서 풀 파인튜닝과의 차이는 학습 방법에서만 나옵니다.

In [ ]:
# 지난 시간 데이터의 시스템 메시지와 글자까지 같은 문장. 생성할 때도 같은 문장이어야 학습한 대로 답합니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_reply(model, question, system_prompt=SYSTEM_PROMPT, max_new_tokens=256):
    """질문 하나를 학습 데이터와 같은 chat template으로 감싸 모델이 새로 만든 답만 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
        enable_thinking=False,       # 빈 생각 블록(<think></think>)을 붙여 생각 과정(thinking) 없이 바로 답합니다
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한. 답이 이보다 길면 끝까지 쓰기 전에 멈춥니다
            do_sample=False,                       # 매번 확률이 가장 높은 토큰을 골라(그리디 생성) 같은 질문에 같은 답이 나옵니다
            eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
            pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
        )

    # 입력 프롬프트 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
from datasets import Dataset

# 지난 시간에 원본 단위로 나눠 저장한 노무 상담 학습, 검증 파일을 그대로 읽습니다
# 행 하나는 question, answer 등을 담은 딕셔너리입니다
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
val_rows = load_jsonl(data_dir / "본실습" / "labor_val.jsonl")

# 지난 시간처럼 prompt(system, user)와 completion(assistant)으로 나눠, 답변 토큰만 손실에 들어가게 합니다
# Dataset.from_list: 딕셔너리 리스트를 SFTTrainer가 받는 Dataset 으로 바꿉니다
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])
print("학습 행 수:", len(train_dataset), "| 검증 행 수:", len(val_dataset))

### 2-4. 학습 전 답

학습 전 모델(Base)의 답은 **어댑터를 붙이기 전에** 모아 둡니다. 어댑터를 붙이면 `base_model`의 선형 층이 그 자리에서 LoRA 층으로 바뀌기 때문입니다.

- **질문:** 여섯 개(학습 2, 검증 1, 범위 밖 1, 일반 2), 지난 시간 교안 01과 같습니다.
- **일반 질문:** 시스템 프롬프트 없이 묻습니다.

In [ ]:
# 상담 범위 밖 질문에 다는 거절 답(지난 시간 데이터와 같은 문장)
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 학습 데이터에서 정답 답변이 거절 답인 첫 행(찾으면 break 로 반복을 멈춥니다)
for row in train_rows:
    if row["answer"] == REFUSAL_ANSWER:
        refusal_row = row
        break

# 노무와 상관없는 상식 질문(정답 답변은 reference_answer)
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 학습 전·후에 똑같이 물을 질문 여섯 개와 그 종류(지난 시간 교안 01과 같은 행)
check_rows = [train_rows[300], train_rows[900], val_rows[39], refusal_row, general_rows[0], general_rows[12]]
check_kinds = ["학습 데이터", "학습 데이터", "검증 데이터", "범위 밖 질문", "일반 질문", "일반 질문"]


def answer_check_questions(model):
    """여섯 질문에 model이 한 답을 같은 순서의 리스트로 돌려줍니다. 일반 질문은 시스템 프롬프트 없이 묻습니다."""
    answers = []
    for row, kind in zip(check_rows, check_kinds):
        if kind == "일반 질문":
            answers.append(generate_reply(model, row["question"], system_prompt=None))
        else:
            answers.append(generate_reply(model, row["question"]))
    return answers

In [ ]:
# 정답 답변: 앞 네 행(노무 학습·검증 데이터)은 answer, 뒤 두 행(일반 질문 파일)은 reference_answer 열에 있습니다
reference_answers = [row["answer"] for row in check_rows[:4]] + [row["reference_answer"] for row in check_rows[4:]]

# 종류, 질문, 정답 답변, Base 답을 한 테이블에 둡니다(학습 뒤 답은 뒤에서 열로 더합니다)
check_table = pd.DataFrame({
    "종류": check_kinds,
    "질문": [row["question"] for row in check_rows],
    "정답 답변": reference_answers,
    "Base 답변": answer_check_questions(base_model),
})
display(check_table)

## 3. 어댑터 설정: LoraConfig

`peft`의 `LoraConfig`는 어댑터를 어느 층에, 어떤 크기로 붙일지 담습니다.

### 3-1. 대상 층: target_modules

`target_modules`는 **어댑터를 붙일 선형 층을 고르는 설정**입니다. 값은 두 가지로 줄 수 있습니다.

| 주는 값 | 무엇인가 | 어댑터가 붙는 곳 |
|---|---|---|
| `"all-linear"` | 층 이름이 아니라 peft가 정해 둔 **약속어** | 출력층 `lm_head`를 뺀 모든 선형 층 |
| `["gate_proj", "up_proj", "down_proj"]` | 선형 층 **이름의 리스트**. 전체 이름(예: `model.layers.0.mlp.gate_proj`)의 마지막 부분만 적습니다 | 전체 이름이 이 이름으로 끝나는 선형 층만(이 예는 24개 디코더 층마다 MLP의 세 선형 층) |

- **출력층 `lm_head`는 빠집니다:** `lm_head`는 마지막 은닉 벡터(1024개 값)를 어휘 248,320개 토큰 각각의 점수로 바꾸는 선형 층입니다. peft는 `"all-linear"`를 받으면 이 출력층만 빼고 나머지 선형 층에 어댑터를 붙입니다. 그래서 선형 층 테이블에서도 `lm_head`는 세지 않습니다.

- **이 교안의 선택:** `"all-linear"`입니다. TRL 문서(지난 시간부터 쓰는 `SFTTrainer`가 들어 있는 Hugging Face 학습 라이브러리 `trl`의 공식 문서)의 [LoRA Without Regret](https://huggingface.co/docs/trl/v1.8.0/en/lora_without_regret) 페이지가 어텐션만이 아니라 MLP까지 모든 선형 층에 어댑터를 붙이기를 권합니다.
- **층 이름은 모델마다 다릅니다:** 이름 리스트를 쓰려면 먼저 모델 구조에서 이름을 확인합니다.
- **이 모델의 구조:** 디코더 층 24개 가운데 18개는 선형 어텐션(`linear_attn`, 계산량을 줄인 어텐션 변형), 6개는 전체 어텐션(`self_attn`, 모든 토큰을 서로 비교하는 일반 어텐션)이고, 모든 층에 MLP(`mlp`)가 있습니다. 어댑터는 그 안의 선형 층(`nn.Linear`, 예: `in_proj_qkv`, `q_proj`, `gate_proj`)에 붙습니다.

<img src="images/03_target_modules.png" width="1050" alt="target_modules는 어댑터를 붙일 선형 층의 이름입니다. 선형 어텐션 층(18개)과 전체 어텐션 층(6개)의 선형 층, MLP의 gate_proj, up_proj, down_proj가 대상이고, lm_head는 all-linear에서도 빠집니다.">

`print(base_model)`로 모델 구조를 출력해 층 이름을 확인합니다. `Linear(in_features=..., out_features=...)`로 나오는 것이 어댑터를 붙일 수 있는 선형 층입니다.

In [ ]:
# 학습 전 모델의 구조: 모듈 이름과 선형 층(Linear)의 입력, 출력 크기
# 들여쓰기가 포함 관계입니다(layers 안에 디코더 층, 그 안에 mlp, 그 안에 gate_proj)
print(base_model)

선형 층을 이름 끝부분(예: `gate_proj`)별로 모아 층 수와 $W$ 모양을 보고, `r=8` 어댑터 파라미터 합 $r \times (d_{out} + d_{in}) \times$ 층 수를 계산합니다.

In [ ]:
lora_rank = 8   # 어댑터의 가운데 차원 r(LoraConfig 에 넣을 값과 같음)

# 1) 모든 모듈을 (이름, 모듈)로 돌며 선형 층(nn.Linear)만 이름 끝부분별로 셉니다
#    이름 예: "model.layers.0.mlp.gate_proj" -> 끝부분 "gate_proj"
linear_counts = {}   # 끝부분 이름 -> 층 수
linear_shapes = {}   # 끝부분 이름 -> W 모양(출력 크기 d_out, 입력 크기 d_in)
for name, module in base_model.named_modules():
    # 출력층 lm_head는 "all-linear"에서도 빠지므로 세지 않습니다
    if isinstance(module, torch.nn.Linear) and name != "lm_head":
        short_name = name.split(".")[-1]
        if short_name not in linear_counts:
            linear_counts[short_name] = 0
            linear_shapes[short_name] = tuple(module.weight.shape)
        linear_counts[short_name] += 1

# 2) 이름마다 한 행: A(r x d_in) + B(d_out x r) = r * (d_out + d_in) 개, 여기에 층 수를 곱합니다
linear_rows = []
for short_name in linear_counts:
    out_features, in_features = linear_shapes[short_name]
    adapter_sum = lora_rank * (out_features + in_features) * linear_counts[short_name]
    linear_rows.append({"층 이름": short_name, "층 수": linear_counts[short_name], "W 모양": linear_shapes[short_name],
                        "어댑터 파라미터 합": adapter_sum})
linear_table = pd.DataFrame(linear_rows)
display(linear_table)

**결과 읽기**
- **층 이름:** 모두 열두 가지입니다. 선형 어텐션의 `out_proj`, `in_proj_qkv`, `in_proj_z`, `in_proj_b`, `in_proj_a`(18층씩), 전체 어텐션의 `q_proj`, `k_proj`, `v_proj`, `o_proj`(6층씩), MLP의 `gate_proj`, `up_proj`, `down_proj`(24층씩)입니다.
- **쓰임:** `target_modules`에 이름 리스트를 줄 때 이 이름을 씁니다. 예를 들어 MLP에만 붙이려면 `["gate_proj", "up_proj", "down_proj"]`를 줍니다.

### 3-2. 어댑터 크기와 배율: r, lora_alpha

| 설정 | 이 교안 값 | 뜻 |
|---|---|---|
| `r` | 8 | $B$와 $A$의 가운데 차원($BA$의 rank 상한). 클수록 학습 파라미터 수가 늘어납니다 |
| `lora_alpha` | 16 | 배율 `lora_alpha / r`을 정하는 값 |
| `lora_dropout` | 0.05 | 어댑터 입력에 거는 드롭아웃 비율(`peft` 기본값 0.0) |
| `task_type` | `"CAUSAL_LM"` | 다음 토큰을 예측하는 생성 모델. `get_peft_model`이 모델을 생성용 클래스(`PeftModelForCausalLM`)로 감쌉니다 |

- **배율(scaling):** 어댑터 출력 $BAx$에 곱하는 `lora_alpha / r`로, 이 교안은 16 / 8 = 2.0입니다. 설정에서 계산한 숫자라 학습하지 않습니다.
- **`r`과 `lora_alpha`를 따로 두는 이유:** `r`은 가운데 차원이라 학습 파라미터 수를 정하고, `lora_alpha`는 배율을 거쳐 어댑터 출력의 세기를 정합니다. 따로 두어야 둘 중 하나만 바꿀 수 있습니다.
- **왜 `r`로 나누나:** $BAx$는 `r`개 값을 더해 만든 출력이라, `r`을 키우면 어댑터 출력도 커집니다. 나누지 않으면 `r`을 바꿀 때마다 같은 학습률이 너무 세거나 약해져, 학습률을 다시 찾아야 합니다. 그래서 `r`로 나눠 `r`이 달라도 어댑터 출력의 크기가 비슷하게 유지되도록 합니다([LoRA 논문](https://arxiv.org/abs/2106.09685) 4.1절).
- **실무:** `lora_alpha`는 처음 고른 `r`의 1~2배로 정해 고정하고, 필요하면 `r`만 바꿉니다.
- **`r` 고르기:** 흔히 8~64입니다. [LoRA Without Regret](https://thinkingmachines.ai/blog/lora/) 실험에서는 흔한 파인튜닝 데이터 크기라면 작은 rank의 어댑터에도 필요한 정보가 다 담겨, rank를 키워도 결과가 거의 같았습니다. 이 단원의 노무 학습 데이터(1,169행)는 그보다도 작습니다.

<img src="images/04_alpha_scale.png" width="1050" alt="배율은 lora_alpha / r입니다. r 8, lora_alpha 16(교안 01)은 2.0, r 32, lora_alpha 16(교안 02)은 0.5로, lora_alpha를 고정하면 r이 커질수록 배율이 작아집니다.">

In [ ]:
from peft import LoraConfig

# 본실습 설정: 출력층을 뺀 모든 선형 층에 r=8 어댑터를 붙이고, 배율은 16 / 8 = 2.0입니다
# LoraConfig 는 설정만 담습니다. 어댑터를 실제로 붙이는 것은 get_peft_model 입니다
lora_config = LoraConfig(
    r=8,                          # B, A의 가운데 차원(BA 의 rank 상한)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # 어댑터 입력에 거는 드롭아웃 비율
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

## 4. 어댑터 붙이기: get_peft_model

### 4-1. 학습 파라미터 수: print_trainable_parameters

`get_peft_model(base_model, lora_config)`은 대상 층에 어댑터를 붙이고 베이스 가중치를 고정한 모델(`lora_model`)을 돌려줍니다. `print_trainable_parameters()`는 학습 파라미터 수, 전체 수, 비율을 출력합니다.

In [ ]:
from peft import get_peft_model

# 대상 선형 층을 LoRA 층(W + A, B)으로 바꿔 끼우고, 베이스 가중치는 requires_grad=False 로 고정합니다
# base_model 도 그 자리에서 바뀌므로, 이제 base_model 의 선형 층에도 어댑터가 붙어 있습니다
lora_model = get_peft_model(base_model, lora_config)

# 학습 파라미터 수, 전체 수, 비율(%)을 한 줄로 출력합니다
lora_model.print_trainable_parameters()

# 같은 두 수를 (학습 수, 전체 수) 튜플로 받아, 선형 층 테이블의 어댑터 파라미터 합과 비교합니다
trainable_count, all_count = lora_model.get_nb_trainable_parameters()
print("선형 층 테이블의 합과 같은가:", trainable_count == linear_table["어댑터 파라미터 합"].sum())

**결과 읽기**
- **trainable params:** 5,411,328개로 선형 층 테이블의 합과 같습니다(True).
- **all params:** 베이스 모델 파라미터에 어댑터를 더한 757,804,352개입니다. 비율 0.7141%는 이 수에 대한 값이고, 베이스 모델(752,393,024개) 대비로는 약 0.72%입니다.

### 4-2. 어댑터가 붙은 자리: base_layer, lora_A, lora_B

첫 디코더 층의 `gate_proj`를 꺼내 봅니다. 경로 `lora_model.base_model.model.model.layers[0]`은 다음 순서로 들어갑니다.

- **`lora_model.base_model`:** 어댑터를 관리하는 `LoraModel`입니다. 이름은 같아도 변수 `base_model`과는 다른 객체입니다.
- **`.model`:** 원래 모델(`Qwen3_5ForCausalLM`)로, 변수 `base_model`이 가리키는 바로 그 객체입니다(어댑터를 붙이며 그 자리에서 바뀌었습니다).
- **`.model`:** 디코더 층들을 담은 본체입니다.

In [ ]:
# 첫 디코더 층의 MLP gate_proj: 원래 선형 층(base_layer) 옆에 어댑터 lora_A, lora_B가 붙었습니다
# 경로: LoraModel(.base_model) -> 원래 모델(.model) -> 본체(.model) -> 디코더 층 0 -> mlp -> gate_proj
gate_layer = lora_model.base_model.model.model.layers[0].mlp.gate_proj
display(gate_layer)

In [ ]:
# named_parameters: 이 층의 (이름, 텐서) 쌍. W(base_layer), A, B만 있고 배율은 없습니다
# A 모양은 (r, d_in), B 모양은 (d_out, r)입니다. requires_grad 가 True 인 텐서만 학습합니다
for name, parameter in gate_layer.named_parameters():
    print(name, tuple(parameter.shape), "| 자료형:", parameter.dtype, "| 학습하는가:", parameter.requires_grad)

# lora_A, lora_B 는 어댑터 이름을 키로 고르는 모듈 묶음입니다(이름을 주지 않으면 "default")
# B 는 0 에서 시작하므로 가장 큰 절댓값도 0 입니다
print("B의 가장 큰 절댓값:", gate_layer.lora_B["default"].weight.abs().max().item())

# 배율은 lora_alpha / r 로 계산해 둔 파이썬 float 하나입니다(scaling 도 어댑터 이름이 키인 딕셔너리)
scaling = gate_layer.scaling["default"]
print("배율(lora_alpha / r):", scaling, "| 자료형:", type(scaling))

**결과 읽기**
- **`base_layer`, `lora_A`, `lora_B`:** 원래 선형 층($W$)은 `base_layer`로 남고 옆에 $A$, $B$가 붙었습니다. `default`는 어댑터 이름으로, 어댑터를 여러 개 얹을 때 이 이름으로 고릅니다(교안 02).
- **빈 항목:** `lora_embedding_A`, `lora_embedding_B`(임베딩 층용)와 `lora_magnitude_vector`(LoRA 변형인 DoRA용, 교안 02)는 이 설정에서 쓰지 않아 비어 있습니다.
- **모양과 처음 값:** $A$는 (8, 1024), $B$는 (3584, 8)로 앞의 계산과 같고, $B$는 0에서 시작합니다. 자료형은 베이스 모델과 같은 float32입니다.
- **배율과 학습 파라미터:** 배율 2.0은 파이썬 `float`라 파라미터 목록(`base_layer.weight`, `lora_A.default.weight`, `lora_B.default.weight`)에 없습니다. `base_layer.weight`는 `requires_grad`가 `False`라 $A$, $B$만 학습합니다.

## 5. 어댑터 학습

### 5-1. LoRA의 학습률: learning_rate

LoRA는 **풀 파인튜닝보다 약 10배 큰 학습률**을 씁니다. 왜 10배인지는 아직 이론으로 다 설명되지 않았고, 실험에서 거듭 확인된 값입니다.

- **근거:** TRL 문서 [PEFT Integration](https://huggingface.co/docs/trl/v1.8.0/en/peft_integration)은 풀 파인튜닝에 2.0e-5, LoRA에 2.0e-4를 권하고, [LoRA Without Regret](https://thinkingmachines.ai/blog/lora/) 실험에서도 가장 좋은 학습률이 풀 파인튜닝의 약 10배였습니다.
- **이 교안:** lr 2e-4
- **쓸 어댑터는 교안 02에서 고름:** 에폭마다 저장한 어댑터 4개 가운데 쓸 것은 교안 02에서 판사가 채점한 정답률로 고릅니다. 그래서 검증 손실이 올라도 멈추지 않고 4에폭을 끝까지 학습합니다.

### 5-2. 학습 설정과 학습: SFTTrainer

설정은 지난 시간 교안 01 에폭 늘리기와 같고 **학습률, 폴더, run 이름만 다릅니다.**

- **저장:** 에폭마다 어댑터가 `checkpoint-<갱신 횟수>` 폴더에 저장됩니다. 갱신 횟수는 배치 하나마다 어댑터를 한 번 고친 누적 횟수입니다(학습기의 `global_step`).
- **기록:** MLflow 실험 `labor-lora`에 남습니다.

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("labor-lora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
from trl import SFTConfig, SFTTrainer

# 지난 시간 에폭 늘리기 설정과 같은 값입니다. 다른 줄에만 '다름'을 적었습니다
sft_config = SFTConfig(
    output_dir=str(output_dir / "labor_lora_lr2e-4"),  # 다름: 에폭마다 checkpoint-<갱신 횟수> 폴더가 생깁니다(교안 02가 불러옵니다)
    learning_rate=2e-4,                         # 다름: LoRA 권장 학습률
    num_train_epochs=4,                         # 중간에 멈추지 않고 끝까지 학습합니다
    per_device_train_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",                      # 에폭이 끝날 때마다 검증 손실을 계산해 기록합니다
    save_strategy="epoch",                      # 에폭이 끝날 때마다 어댑터를 저장합니다
    save_only_model=True,                       # 옵티마이저(AdamW) 상태는 빼고 모델 쪽(어댑터)만 저장합니다
    logging_steps=50,
    eval_on_start=True,                         # 학습 전(0에폭)에도 검증 손실을 계산합니다
    report_to=["mlflow"],
    run_name="lora_lr_2e-4",                    # 다름
)

# 학습 뒤 베이스 가중치가 그대로인지 보려고, 학습 전 gate_proj 의 W를 복사해 둡니다
# detach().clone(): 계산 기록에서 떼어 낸 별도 복사본이라, 원본이 바뀌어도 이 값은 그대로입니다
gate_weight_before = gate_layer.base_layer.weight.detach().clone()

# 어댑터를 붙인 모델을 넘기면 SFTTrainer가 어댑터만 갱신합니다(requires_grad=True 인 A, B만 기울기를 받음)
trainer = SFTTrainer(
    model=lora_model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
# 데이터 파일, 행 수, LoRA 설정을 run에 남기고 학습합니다. 갱신 횟수 = 올림(1169 / 4) * 4 = 293 * 4 = 1172
with mlflow.start_run(run_name=sft_config.run_name):
    mlflow.log_params({"train_file": "labor_train.jsonl", "train_rows": len(train_dataset),
                       "val_file": "labor_val.jsonl", "val_rows": len(val_dataset),
                       "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "target_modules": "all-linear"})
    trainer.train()

### 5-3. 학습 곡선: log_history

학습이 정상으로 진행됐는지 학습 손실과 검증 손실로 봅니다. 이 곡선은 학습 기록이고, 어댑터를 고르는 기준이 아닙니다.

In [ ]:
# log_history: 기록마다 딕셔너리 하나. 학습 기록 줄에는 loss, 검증 기록 줄에는 eval_loss 가 있습니다
# 다른 쪽 칸은 NaN 이라 dropna 로 학습 손실 줄과 검증 손실 줄을 나눕니다
log_table = pd.DataFrame(trainer.state.log_history)
train_log = log_table.dropna(subset=["loss"])
eval_log = log_table.dropna(subset=["eval_loss"])

# 학습 손실(파랑)과 검증 손실(빨강)을 에폭 축에 그립니다. 검증 손실은 학습 전(0에폭)부터 있습니다
plt.figure(figsize=(7, 4))
plt.plot(train_log["epoch"], train_log["loss"], marker="o", color="#2563eb", label="학습 손실(loss)")
plt.plot(eval_log["epoch"], eval_log["eval_loss"], marker="o", color="#dc2626", label="검증 손실(eval_loss)")
plt.title("LoRA 학습 곡선 (lr 2e-4)")
plt.xlabel("에폭 (epoch)")
plt.ylabel("손실 (크로스 엔트로피)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**그래프 읽기**
- **검증 손실:** 학습 전 2.83에서 1에폭 1.58, 2에폭 1.52(가장 낮음)로 내려간 뒤 3에폭 1.56, 4에폭 1.64로 오릅니다.
- **풀 파인튜닝과 비교:** 풀 파인튜닝(1~2에폭 1.43, 4에폭 1.54)보다 조금 높고, 2에폭 뒤 오르는 모양은 같습니다.
- **학습 손실:** 새 에폭마다 이미 본 행을 다시 보므로 에폭 경계마다 계단처럼 떨어져, 4에폭 끝에 약 0.18입니다. 학습 손실은 내려가는데 검증 손실은 오르므로, 2에폭 뒤로는 학습 데이터를 외우는 쪽으로 학습합니다.
- **다시 실행하면:** 같은 설정이어도 GPU의 계산 순서 차이로 손실은 소수 둘째 자리에서, 뒤의 답 일부는 글자가 조금씩 달라질 수 있습니다.
- **어댑터는 버리지 않음:** 검증 손실이 오른 에폭의 어댑터도 남깁니다. 어느 어댑터의 답이 정답 답변에 더 가까운지는 손실이 아니라, 교안 02에서 답을 채점해 정합니다.

### 5-4. 베이스 가중치 확인: torch.equal

학습 전에 복사해 둔 `gate_proj`의 $W$를 학습 뒤 $W$와 비교하고, $B$가 0에서 벗어났는지 봅니다.

In [ ]:
# torch.equal: 두 텐서의 모양과 모든 값이 같으면 True. B는 0이 아닌 값을 배웠는지 봅니다
print("베이스 가중치가 학습 전과 같은가:", torch.equal(gate_weight_before, gate_layer.base_layer.weight))
print("학습 뒤 B의 가장 큰 절댓값:", gate_layer.lora_B["default"].weight.abs().max().item())

**결과 읽기**
- 베이스 가중치는 학습 전과 같고(True), $B$의 가장 큰 절댓값은 약 0.018로 0에서 벗어났습니다. 바뀐 것이 어댑터뿐이므로 저장할 것도 어댑터뿐입니다.

### 5-5. 학습 전·후 답

같은 질문 여섯 개를 같은 방법(그리디 생성)으로 LoRA 모델(4에폭 끝)에 물어 테이블에 열로 더합니다.

- **평가 모드:** 모델에는 학습 모드와 평가 모드가 있고, 학습 모드에서만 드롭아웃(학습 중 일부 값을 무작위로 0으로 만드는 것, `lora_dropout` 포함)이 켜집니다. 답을 만들기 전에 `lora_model.eval()`로 평가 모드로 바꿔 드롭아웃을 끕니다.

In [ ]:
# 드롭아웃을 끄고(평가 모드), 같은 여섯 질문을 LoRA 모델에 물어 Base 답이 있는 테이블에 열로 더합니다
lora_model.eval()
check_table["LoRA 답변"] = answer_check_questions(lora_model)
display(check_table)

**결과 읽기**(Base 답과 LoRA 답)
- **학습 데이터:** Base는 배우자 유산 휴가를 '1년 또는 2년', 조정안 기한을 '20일 이내'로 틀렸습니다. 학습 뒤에는 둘 다 정답 답변과 글자까지 같습니다(5일 범위·최초 3일 유급, 60일 이내).
- **검증 데이터:** 학습에 없던 휴게시간 질문에 Base는 '1시간'으로 답합니다. 학습 뒤에는 노무 답 말투('근로기준법 제50조에 따라')는 익혔지만 조문과 답이 틀립니다. 제50조는 근로시간 조문이고, 4시간 근로를 묻는데 '8시간 근로하면 1시간 이상'으로 답합니다(정답은 제54조의 30분 이상).
- **범위 밖 질문:** Base는 연말정산 일정을 지어 답하고, 학습 뒤에는 정해진 거절 문장으로 답합니다.
- **일반 질문:** 둘 다 서울과 광합성을 맞힙니다. 학습 뒤에는 한 문장으로 짧아졌고, '대한민국의 수도은'처럼 조사가 하나 어긋났습니다.
- **주의:** 질문 여섯 개라 경향만 보입니다. 쓸 어댑터는 교안 02에서 판사가 채점한 정답률로 고릅니다.

### 5-6. 어댑터 끄기: disable_adapter

LoRA는 베이스 가중치를 그대로 두므로, **학습 전 모델을 따로 불러 두지 않아도 Base로 답할 수 있습니다.**

- **`with lora_model.disable_adapter():`** 블록 안에서는 어댑터가 꺼지고, 블록을 나오면 다시 켜집니다.
- **확인:** 첫 질문을 다시 물어 Base 답, LoRA 답과 비교합니다.

In [ ]:
# with 블록 안에서만 어댑터를 끄고 첫 질문을 묻습니다
question = check_rows[0]["question"]
with lora_model.disable_adapter():
    adapter_off_answer = generate_reply(lora_model, question)

# 블록을 나온 뒤에는 어댑터가 다시 켜진 상태로 같은 질문을 묻습니다
# check_table.loc[0, 열 이름]: 테이블 0번 행(첫 질문)의 그 열 답
adapter_on_answer = generate_reply(lora_model, question)
print("어댑터 끔 답이 Base 답과 같은가:", adapter_off_answer == check_table.loc[0, "Base 답변"])
print("블록 밖의 답이 LoRA 답과 같은가:", adapter_on_answer == check_table.loc[0, "LoRA 답변"])

**결과 읽기**
- 두 비교 모두 True입니다. 같은 모델 객체가 블록 안에서는 Base로, 밖에서는 학습한 어댑터로 답합니다.

## 6. 어댑터 저장과 다시 불러오기

### 6-1. 체크포인트 폴더의 어댑터 설정: adapter_config.json

체크포인트는 학습 중간에 저장한 결과입니다. LoRA 체크포인트는 **어댑터 폴더 하나**이고, 베이스 모델은 들어 있지 않습니다.

- **위치:** `output/labor_lora_lr2e-4/checkpoint-<갱신 횟수>`. 갱신 횟수는 에폭마다 올림(학습 행 수 / 배치) = 293씩 늘어 293, 586, 879, 1172입니다(교안 02가 이 4개를 불러옵니다).
- **쓰는 법:** 같은 베이스 모델을 먼저 불러와 그 위에 어댑터를 얹습니다. 어느 베이스 모델인지는 어댑터 설정 파일 `adapter_config.json`에 적혀 있습니다.

<img src="images/05_adapter_files.png" width="1050" alt="어댑터 폴더에는 설정과 학습한 A·B 값만 있고 베이스 모델은 없습니다. 쓸 때는 베이스 모델을 불러와 어댑터를 얹습니다.">

In [ ]:
# 마지막 체크포인트 폴더: 마지막 갱신 횟수(trainer.state.global_step)가 폴더 이름입니다
last_checkpoint_dir = output_dir / "labor_lora_lr2e-4" / f"checkpoint-{trainer.state.global_step}"

# 어댑터 설정 파일에서 다시 불러올 때 쓰는 항목만 봅니다
with open(last_checkpoint_dir / "adapter_config.json", encoding="utf-8") as f:
    adapter_config = json.load(f)
display({key: adapter_config[key] for key in ["base_model_name_or_path", "peft_type", "r", "lora_alpha", "lora_dropout", "target_modules", "use_dora"]})

**결과 읽기**
- **`base_model_name_or_path`:** `Qwen/Qwen3.5-0.8B`입니다. 다시 불러올 때 이 모델 위에 얹습니다.
- **`target_modules`:** `"all-linear"` 대신 실제로 어댑터를 붙인 층 이름 열두 가지(선형 층 테이블과 같음)가 저장됩니다.

### 6-2. 어댑터 저장: save_pretrained

학습기 없이 어댑터만 저장할 때는 **`save_pretrained(폴더)` 함수**를 씁니다. 학습기 체크포인트도 이 함수로 저장하므로 파일 이름이 같습니다. `output/labor_lora_adapter`에 저장하고 마지막 체크포인트 폴더와 파일을 비교합니다.

In [ ]:
# 어댑터 설정과 학습한 A, B 값만 저장합니다(베이스 모델은 저장하지 않습니다)
lora_model.save_pretrained(output_dir / "labor_lora_adapter")

# 체크포인트 폴더와 save_pretrained 폴더의 파일 이름과 크기를 한 테이블로 봅니다
# st_size 는 바이트 단위라 1024 ** 2 로 나눠 MB 로 바꿉니다
file_rows = []
for folder in [last_checkpoint_dir, output_dir / "labor_lora_adapter"]:
    for path in sorted(folder.iterdir()):
        file_rows.append({"폴더": folder.name, "파일": path.name, "크기(MB)": round(path.stat().st_size / 1024 ** 2, 2)})
display(pd.DataFrame(file_rows))

**결과 읽기**
- **`adapter_model.safetensors`:** 학습한 $A$, $B$ 값으로, 두 폴더 모두 20.7MB입니다(파라미터 5,411,328개 × float32 4바이트 = 약 20.6MB, 나머지는 텐서 이름과 모양을 적은 파일 앞부분). 풀 파인튜닝 체크포인트는 모델 전체라 약 3GB였습니다.
- **save_pretrained 폴더:** `adapter_config.json`, `adapter_model.safetensors`, `README.md` 세 개뿐입니다.
- **체크포인트 폴더의 나머지:** 토크나이저 파일(`tokenizer.json`, `tokenizer_config.json`, `chat_template.jinja`), 학습 기록 `trainer_state.json`, 학습 설정 `training_args.bin`입니다. 학습기가 함께 저장할 뿐 어댑터를 얹는 데는 쓰지 않습니다.

### 6-3. 어댑터 다시 불러오기: PeftModel.from_pretrained

**`PeftModel.from_pretrained(베이스 모델, 폴더)` 함수**는 `adapter_config.json`대로 어댑터 층을 만들고 저장한 $A$, $B$ 값을 채운 모델을 돌려줍니다.

- **기본 상태:** `is_trainable` 기본값이 `False`라 어댑터는 학습하지 않는 상태로 얹히고, 모델은 평가 모드가 됩니다.
- **확인:** 새 베이스 모델에 `output/labor_lora_adapter`를 얹어, 첫 질문의 답이 저장 전과 같은지 봅니다.

In [ ]:
from peft import PeftModel

# 어댑터가 없는 베이스 모델을 새로 불러오고, 저장한 어댑터 폴더를 그 위에 얹습니다
# from_pretrained 는 adapter_config.json 대로 LoRA 층을 만들고 adapter_model.safetensors 의 A, B 값을 채웁니다
reload_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
reloaded_model = PeftModel.from_pretrained(reload_base, output_dir / "labor_lora_adapter")

# 첫 질문의 답을 저장 전 LoRA 모델의 답(check_table)과 비교합니다
reloaded_answer = generate_reply(reloaded_model, check_rows[0]["question"])
print("저장 전 LoRA 답과 같은가:", reloaded_answer == check_table.loc[0, "LoRA 답변"])

**결과 읽기**
- True입니다. 폴더 하나와 같은 베이스 모델만 있으면 어댑터를 다시 쓸 수 있습니다. 교안 02는 이 방법으로 어댑터 4개를 얹습니다.

본실습이 끝났으니 학습기와 모델을 지워 GPU 메모리를 비웁니다(어댑터는 저장해 두었습니다).

In [ ]:
import gc

# 본실습의 학습기, 모델, 모델 안을 가리키는 변수를 지웁니다(한 번만 실행)
# module, parameter 는 앞 반복문이 마지막으로 가리킨 층과 파라미터입니다
del trainer, lora_model, base_model, module, gate_layer, parameter, gate_weight_before
del reloaded_model, reload_base
gc.collect()              # 더 이상 쓰지 않는 객체를 메모리에서 정리합니다
torch.cuda.empty_cache()  # PyTorch가 잡아 두었던 빈 GPU 메모리를 돌려줍니다

## 7. 🖐️ 함께 따라하기: 전월세 상담 어댑터 학습하기

본실습의 LoRA 학습을 **전월세 상담 데이터**에 적용해 어댑터를 학습하고 저장합니다. 평가, 그리고 노무 어댑터와 바꿔 끼워 쓰기는 교안 02 함께 따라하기에서 합니다.

**본실습과 다른 점**
- **데이터:** `data/따라하기/lease_train.jsonl`, `lease_val.jsonl`(지난 시간 풀 파인튜닝한 주택임대차보호법 질문-답변과 같음)
- **시스템 프롬프트:** `SYSTEM_PROMPT_LEASE`(학습 데이터의 시스템 메시지와 같은 문장, 답을 만들 때도 넘김)
- **기록 간격:** 학습 행이 137개라 4에폭 동안 모두 140번만 갱신합니다. 그래서 학습 손실을 기록하는 간격 `logging_steps`를 10으로 줄입니다.
- **저장:** 중간 체크포인트 없이(`save_strategy="no"`) 학습 뒤 `save_pretrained`로 `output/lease_lora_lr2e-4`에 저장합니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다(어댑터 설정과 붙이기, 학습 설정, 저장). 먼저 직접 작성해 봅니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

**설정**
- **본실습과 같음:** `LoraConfig`(`r=8`, `lora_alpha=16`, `target_modules="all-linear"`, `lora_dropout=0.05`), lr 2e-4, 에폭 4, 배치 4, `max_length=1024`, `eval_strategy="epoch"`, `eval_on_start=True`, `seed=0`, `gradient_checkpointing=False`
- **기록:** run 이름 `lease_lora_lr_2e-4`, MLflow 실험 `lease-lora`(본실습 run과 섞이지 않게)

### 7-1. 데이터 읽기

In [ ]:
# 전월세 데이터의 시스템 메시지와 글자까지 같은 문장
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."

# 지난 시간 따라하기에서 원본 단위로 나눠 저장한 전월세 학습, 검증 파일을 읽고 Dataset으로 만듭니다
lease_train_rows = load_jsonl(data_dir / "따라하기" / "lease_train.jsonl")
lease_val_rows = load_jsonl(data_dir / "따라하기" / "lease_val.jsonl")
lease_train_dataset = Dataset.from_list([to_prompt_completion(row) for row in lease_train_rows])
lease_val_dataset = Dataset.from_list([to_prompt_completion(row) for row in lease_val_rows])
print("학습 행 수:", len(lease_train_dataset), "| 검증 행 수:", len(lease_val_dataset))

### 7-2. 어댑터 붙이기: get_peft_model

In [ ]:
# 학습 전 상태의 모델을 새로 불러옵니다(본실습 어댑터와 따로 학습합니다)
lease_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# 🖐️ 함께 따라하기: 본실습과 같은 LoRA 설정을 만들고, lease_base 에 어댑터를 붙여 lease_model 을 만드세요
lease_lora_config = LoraConfig(
    r=...,
    lora_alpha=...,
    target_modules=...,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
lease_model = ...
lease_model.print_trainable_parameters()

### 7-3. 학습: SFTTrainer

In [ ]:
# 🖐️ 함께 따라하기: 위 '설정'의 값으로 학습률, 에폭 수, 저장 방식을 채우세요
lease_config = SFTConfig(
    output_dir=str(output_dir / "runs" / "lease_lora_lr2e-4"),  # 학습기가 기록을 둘 폴더(체크포인트는 저장하지 않습니다)
    learning_rate=...,                 # 본실습과 같은 학습률
    num_train_epochs=...,              # 본실습과 같은 에폭 수
    per_device_train_batch_size=4,
    seed=0,
    gradient_checkpointing=False,
    max_length=1024,
    eval_strategy="epoch",
    save_strategy=...,                 # 중간 체크포인트를 저장하지 않습니다
    logging_steps=10,                  # 갱신 횟수가 적어 기록 간격을 줄입니다
    eval_on_start=True,
    report_to=["mlflow"],
    run_name="lease_lora_lr_2e-4",
)
lease_trainer = SFTTrainer(
    model=lease_model,
    args=lease_config,
    train_dataset=lease_train_dataset,
    eval_dataset=lease_val_dataset,
    processing_class=tokenizer,
)

# 전월세 run 은 따로 둔 실험에 남깁니다(교안 02 는 준비 셀에서 본실습 실험으로 돌아갑니다)
mlflow.set_experiment("lease-lora")

# 데이터 파일과 행 수를 run에 남기고 학습합니다(본실습과 같은 방법)
with mlflow.start_run(run_name=lease_config.run_name):
    mlflow.log_params({"train_file": "lease_train.jsonl", "train_rows": len(lease_train_dataset),
                       "val_file": "lease_val.jsonl", "val_rows": len(lease_val_dataset)})
    lease_trainer.train()

### 7-4. 어댑터 저장: save_pretrained

전월세 어댑터는 교안 02 함께 따라하기에서 채점하므로 여기서는 저장까지만 합니다.

In [ ]:
# 🖐️ 함께 따라하기: 학습한 전월세 어댑터를 output/lease_lora_lr2e-4 폴더에 저장하세요(교안 02 따라하기가 불러옵니다)
...

# 교안 02가 불러올 어댑터 파일이 생겼는지 봅니다
print("어댑터 파일이 있는가:", (output_dir / "lease_lora_lr2e-4" / "adapter_model.safetensors").exists())

## 8. 핵심 정리

- **LoRA:** $W$는 고정하고, 바뀔 양을 작은 두 행렬의 곱 $BA$로 학습합니다. 출력은 $h = Wx + \frac{\alpha}{r}BAx$이고, $B$가 0에서 시작해 붙인 직후에는 베이스 모델과 같습니다. 층 하나의 학습 파라미터 수는 $r \times (d_{out} + d_{in})$입니다(`gate_proj`, r 8: 36,864개, $W$의 약 1.0%).
- **언제 LoRA를 쓰나:** GPU 메모리가 적거나 분야가 여럿이거나 질문-답변 SFT일 때. 새 지식을 대량으로 넣는 큰 학습이면 풀 파인튜닝.
- **설정:** `r=8`, `lora_alpha=16`(배율 2.0), `target_modules="all-linear"`(`lm_head` 제외), `lora_dropout=0.05`. 학습 파라미터는 5,411,328개(베이스 모델의 약 0.72%)입니다.
- **`r`과 `lora_alpha`:** `r`은 $B$와 $A$의 가운데 차원으로 학습 파라미터 수를 정하고, `lora_alpha`는 배율 `lora_alpha / r`로 어댑터 출력의 세기를 정합니다. 배율은 파이썬 `float`라 학습하지 않습니다. 배율에서 `r`로 나누므로 `r`을 바꿔도 어댑터 출력의 크기가 비슷하게 유지됩니다. 그래서 실무에서는 `lora_alpha`를 처음 고른 `r`의 1~2배로 정해 두고 `r`만 바꿉니다.
- **학습:** `get_peft_model`로 감싼 모델을 같은 `SFTTrainer`에 넘기면 학습기가 어댑터만 갱신합니다. 학습률은 풀 파인튜닝의 약 10배(lr 2e-4)로 4에폭 학습하고, 에폭마다 저장했습니다(갱신 횟수 293, 586, 879, 1172).
- **같은 질문으로 비교:** 학습 데이터 질문은 정답 답변과 같아지고, 범위 밖 질문은 거절하며, 일반 질문은 맞히되 짧아집니다. 학습에 없는 휴게시간 질문은 말투만 따라가고 조문과 숫자는 틀립니다.
- **어댑터 끄기:** `with lora_model.disable_adapter():` 안에서는 같은 모델이 Base로 답하므로, 학습 전 모델을 따로 불러 두지 않아도 됩니다.
- **저장과 다시 불러오기:** 어댑터 폴더(20.7MB, 풀 파인튜닝 체크포인트는 약 3GB)에는 베이스 모델이 없고, `adapter_config.json`에 베이스 모델 이름이 있습니다. 저장은 `save_pretrained`, 다시 쓸 때는 같은 베이스 모델을 불러와 `PeftModel.from_pretrained(베이스 모델, 폴더)`로 얹습니다.

교안 02에서는 어댑터 4개를 지난 시간과 같은 지표로 채점해 쓸 어댑터를 고르고, 풀 파인튜닝 모델과 비교합니다.